In [33]:
# !pip install langchain[azure]
# !pip install --upgrade langchain
# !pip install -U azure-search-documents
# !pip install --upgrade --quiet  azure-identity

https://python.langchain.com/v0.2/docs/integrations/vectorstores/azuresearch/

In [1]:
import os  
# from langchain_community.retrievers import AzureAISearchRetriever
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_openai import AzureOpenAIEmbeddings
from langchain_openai import AzureChatOpenAI
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from azure.search.documents.indexes.models import SimpleField, SearchableField, SearchField, SearchFieldDataType

In [2]:
# Set up Azure OpenAI credentials
os.environ["AZURE_OPENAI_API_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"] = "" 

# Option 2: use an Azure OpenAI account with a deployment of an embedding model
azure_openai_api_version: str = ""
azure_deployment: str = ""

# Set up Azure AI Search credentials
os.environ["AZURE_AI_SEARCH_ENDPOINT"] = ""
os.environ["AZURE_AI_SEARCH_KEY"] = ""
os.environ["AZURE_AI_SEARCH_INDEX_NAME"] = ""

In [3]:
embeddings: AzureOpenAIEmbeddings = AzureOpenAIEmbeddings(
    azure_deployment=azure_deployment,
    openai_api_version=azure_openai_api_version,
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

In [57]:
# # Initialize Azure AI Search vector store
# vector_store = AzureSearch(
#     azure_search_endpoint=os.environ["AZURE_AI_SEARCH_ENDPOINT"],
#     azure_search_key=os.environ["AZURE_AI_SEARCH_KEY"],
#     index_name=os.environ["AZURE_AI_SEARCH_INDEX_NAME"],
#     embedding_function=embeddings.embed_query,
# )

In [64]:
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile",
    ),
    SearchableField(
        name="metadata",
        type=SearchFieldDataType.String,
        searchable=True,
    ),
    # Additional field to store the title
    SearchableField(
        name="title",
        type=SearchFieldDataType.String,
        searchable=True,
    ),
    # Additional field for filtering on document source
    SimpleField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True,
    ),
]

index_name: str = "langchain-aisearch-vectordb"

In [65]:
# Initialize Azure AI Search vector store
vector_store = AzureSearch(
    azure_search_endpoint=os.environ["AZURE_AI_SEARCH_ENDPOINT"],
    azure_search_key=os.environ["AZURE_AI_SEARCH_KEY"],
    index_name=index_name,
    embedding_function=embeddings.embed_query,
    fields=fields)

In [66]:
# Load and process documents
loader = TextLoader("Documents\OPENAI_Terms_of_use.txt")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)


# Add documents to the vector store
vector_store.add_documents(documents=docs)

# retriever = AzureAISearchRetriever(
#     content_key="content", top_k=1, index_name="azureaisearch-rimanshu"
# )

# Create a retrieval chain
retriever = vector_store.as_retriever()


In [67]:
# Initialize Azure OpenAI Chat model
llm = AzureChatOpenAI(
    openai_api_version="",
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
)

# model = AzureChatOpenAI(
#     deployment_name="gpt-4o-mini",
#     model_name="gpt-4o-mini",
#     api_key ="b414c391b44f483c986eda5f05b579b3",
#     azure_endpoint = "https://openai-rimanshu.openai.azure.com/",
#     api_version = "2023-03-15-preview",
# )

In [68]:
prompt_template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

In [70]:
chain_type_kwargs = {"prompt": PROMPT}
qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=retriever, 
                                 chain_type_kwargs=chain_type_kwargs)

# Example usage
query = "Please provide me the summary of the document"
result = qa.invoke(query)
print(result)

{'query': 'Please provide me the summary of the document', 'result': 'The document outlines terms related to arbitration, copyright complaints, and general terms of service. \n\n1. **Severability**: If any part of the arbitration terms is deemed illegal or unenforceable, the rest will still apply, except if it leads to class actions, in which case the entire section becomes unenforceable.\n\n2. **Copyright Complaints**: It details the process for reporting copyright infringement, including the required information for a written claim. This includes a signature, description of the work and alleged infringement, and personal contact details.\n\n3. **General Terms**: It states that rights or obligations under the terms cannot be assigned or transferred by users. OpenAI can assign its rights to affiliates or successors. It encourages informal dispute resolution prior to legal action, allowing 60 days to resolve issues before arbitration can be initiated. \n\nOverall, the document establish

In [ ]:
# from langchain_community.tools import 

from langchain_experimental.utilities import 

In [71]:
!pip install langchain_experimental

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 18.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.2.15
    Uninstalling langchain-community-0.2.15:
      Successfully uninstalled langchain-community-0.2.15
